# model_experiment_LightGBM

მიზანი: Walmart Weekly Sales ამოცანისთვის LightGBM tree-based მოდელის დატრენინგება, time-series validation-ით შეფასება, W&B-ზე დალოგვა და საუკეთესო pipeline-ის artifact-ად შენახვა.

ეს notebook არის შენი ძირითადი ძლიერი მოდელი.

In [1]:
# Run once if packages are missing:
# !pip install -r requirements.txt

In [2]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import pandas as pd
import numpy as np
import joblib
import wandb

from src.config import WANDB_ENTITY, WANDB_PROJECT, MODELS_DIR, SUBMISSIONS_DIR, VALIDATION_FOLDS, VALIDATION_WEEKS, RANDOM_STATE
from src.data_loading import load_raw_data, describe_dataframes, make_submission_frame
from src.validation import evaluate_time_folds, summarize_cv
from src.metrics import wmae
from src.wandb_utils import safe_wandb_init, save_and_log_model, log_dataframe_as_artifact

pd.set_option('display.max_columns', 100)
MODELS_DIR.mkdir(exist_ok=True, parents=True)
SUBMISSIONS_DIR.mkdir(exist_ok=True, parents=True)
print('W&B entity:', WANDB_ENTITY)
print('W&B project:', WANDB_PROJECT)

from src.models import WalmartSalesForecaster, make_lightgbm_model

W&B entity: gchal22-free-university-of-tbilisi-
W&B project: store_sales_forecast


In [3]:
# Login opens browser/token prompt if needed.
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\West\_netrc.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [4]:
data = load_raw_data()
describe_dataframes(data)

,name,rows,columns,column_names
0,train,421570,5,"Store, Dept, Date, Weekly_Sales, IsHoliday"
1,test,115064,4,"Store, Dept, Date, IsHoliday"
2,features,8190,12,"Store, Date, Temperature, Fuel_Price, MarkDown..."
3,stores,45,3,"Store, Type, Size"
4,sample_submission,115064,2,"Id, Weekly_Sales"


In [5]:
train = data['train'].copy()
features = data['features'].copy()
stores = data['stores'].copy()
test = data['test'].copy()
sample_submission = data['sample_submission'].copy()

print(train['Date'].min(), train['Date'].max())
print(test['Date'].min(), test['Date'].max())
print(train.head())

2010-02-05 2012-10-26
2012-11-02 2013-07-26
   Store  Dept        Date  Weekly_Sales  IsHoliday
0      1     1  2010-02-05      24924.50      False
1      1     1  2010-02-12      46039.49       True
2      1     1  2010-02-19      41595.55      False
3      1     1  2010-02-26      19403.54      False
4      1     1  2010-03-05      21827.90      False


## Validation strategy

არ ვიყენებთ random split-ს, რადგან ეს time-series ამოცანაა. ვიყენებთ expanding-window chronological split-ს: წინა თარიღები train, მომდევნო 8 კვირა validation.

In [6]:
FAST_DEV_RUN = False  # True only for quick debugging.
if FAST_DEV_RUN:
    train_for_cv = train[train['Store'].isin([1, 2, 3])].copy()
else:
    train_for_cv = train.copy()

lightgbm_params = {
    'n_estimators': 1200,
    'learning_rate': 0.035,
    'num_leaves': 128,
    'min_child_samples': 80,
    'subsample': 0.85,
    'colsample_bytree': 0.85,
    'reg_alpha': 0.05,
    'reg_lambda': 0.2,
}

def forecaster_factory():
    model = make_lightgbm_model(random_state=RANDOM_STATE, **lightgbm_params)
    return WalmartSalesForecaster(model=model, model_name='LightGBM')

In [7]:
with safe_wandb_init(
    run_name='LightGBM_CV',
    group='LightGBM',
    job_type='cross_validation',
    config={
        'model': 'LightGBM',
        'validation_folds': VALIDATION_FOLDS,
        'validation_weeks': VALIDATION_WEEKS,
        'fast_dev_run': FAST_DEV_RUN,
        **lightgbm_params,
    },
) as run:
    cv_results = evaluate_time_folds(
        forecaster_factory,
        train_for_cv,
        features,
        stores,
        n_folds=VALIDATION_FOLDS,
        validation_weeks=VALIDATION_WEEKS,
        log_callback=wandb.log,
    )
    summary = summarize_cv(cv_results)
    wandb.log(summary)
    wandb.log({'cv_results': wandb.Table(dataframe=cv_results)})
    log_dataframe_as_artifact(run, cv_results, 'lightgbm-cv-results', 'validation_results', 'reports/lightgbm_cv_results.csv')

cv_results

wandb: Currently logged in as: gchal22 (gchal22-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


USING FIXED evaluate_time_folds v2
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.019231 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7662
[LightGBM] [Info] Number of data points in the train set: 350595, number of used features: 59
[LightGBM] [Info] Start training from score 7711.815918
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.070849 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7672
[LightGBM] [Info] Number of data points in the train set: 374203, number of used features: 59
[LightGBM] [Info] Start training from score 7710.935547
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.042688 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7679
[LightGBM] [I

wandb: WARNING Artifact "lightgbm-cv-results" already exists with the same content. No new version will be created.


cv_mae_mean,▁
cv_wmae_mean,▁
cv_wmae_std,▁
fold_1_fold,▁
fold_1_mae,▁
fold_1_train_rows,▁
fold_1_valid_rows,▁
fold_1_wmae,▁
fold_2_fold,▁
fold_2_mae,▁
+11,...


,fold,train_start,train_end,valid_start,valid_end,train_rows,valid_rows,wmae,mae,holiday_wmae_part
0,1,2010-02-05,2012-05-11,2012-05-18,2012-07-06,350595,23608,1521.779985,1521.779985,NaN
1,2,2010-02-05,2012-07-06,2012-07-13,2012-08-31,374203,23638,1546.892059,1546.892059,NaN
2,3,2010-02-05,2012-08-31,2012-09-07,2012-10-26,397841,23729,1355.978102,1352.445684,1363.043237


In [8]:
summary

{'cv_wmae_mean': 1474.8833821969174,
 'cv_wmae_std': 84.70144913683893,
 'cv_mae_mean': 1473.7059094194728}

## Train final LightGBM pipeline on full train

ეს saved object თვითონ აკეთებს merge-ს, feature engineering-ს, lag/rolling features-ს და raw `test.csv`-ზე პროგნოზს.

In [9]:
REGISTER_AS_BEST = True  # Keep True if LightGBM is currently your best model after CV comparison.

final_model = WalmartSalesForecaster(
    model=make_lightgbm_model(random_state=RANDOM_STATE, **lightgbm_params),
    model_name='LightGBM',
)

with safe_wandb_init(
    run_name='LightGBM_Final_Model',
    group='LightGBM',
    job_type='final_training',
    config={'model': 'LightGBM', **lightgbm_params},
) as run:
    final_model.fit(train, features, stores)
    model_path = MODELS_DIR / 'lightgbm_pipeline.joblib'
    metadata = {
        'architecture': 'LightGBM',
        'metric': 'WMAE',
        'uses_raw_test': True,
        'feature_columns': final_model.feature_columns_,
        'notes': 'Recursive lag/rolling feature pipeline trained on full train.csv',
    }
    save_and_log_model(run, final_model, 'lightgbm-pipeline', model_path, metadata=metadata, aliases=['latest', 'candidate'])
    if REGISTER_AS_BEST:
        save_and_log_model(run, final_model, 'best-model', model_path, metadata=metadata, aliases=['latest', 'production'])

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.051961 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7685
[LightGBM] [Info] Number of data points in the train set: 421570, number of used features: 59
[LightGBM] [Info] Start training from score 7680.639160


## Generate LightGBM submission

ეს submission შეგიძლია პირდაპირ ატვირთო Kaggle-ზე. საბოლოოდ მაინც `model_inference.ipynb` გამოიყენე, რადგან იქ best-model artifact იტვირთება W&B-დან.

In [10]:
preds = final_model.predict(test)
submission = make_submission_frame(test, preds, sample_submission)
submission_path = SUBMISSIONS_DIR / 'submission_lightgbm.csv'
submission.to_csv(submission_path, index=False)
submission.head(), submission_path

(               Id  Weekly_Sales
 0  1_1_2012-11-02  35789.862549
 1  1_1_2012-11-09  24525.000149
 2  1_1_2012-11-16  21901.506560
 3  1_1_2012-11-23  24862.353540
 4  1_1_2012-11-30  27561.253627,
 WindowsPath('C:/walmart/submissions/submission_lightgbm.csv'))

In [11]:
with safe_wandb_init('LightGBM_Submission', 'LightGBM', 'submission') as run:
    artifact = log_dataframe_as_artifact(run, submission, 'lightgbm-submission', 'submission', submission_path)
    wandb.log({
        'prediction_min': float(submission['Weekly_Sales'].min()),
        'prediction_max': float(submission['Weekly_Sales'].max()),
        'prediction_mean': float(submission['Weekly_Sales'].mean()),
    })

prediction_max,▁
prediction_mean,▁
prediction_min,▁
prediction_max,223948.49012
prediction_mean,16784.24574
prediction_min,0
